### Document Analysis using LLMs with Python

In [1]:
import pdfplumber
from dotenv import load_dotenv
import os

load_dotenv()  # loads from .env file in the project root

hf_token = os.getenv("HF_TOKEN")
headers = {"Authorization": f"Bearer {hf_token}"}

In [2]:
#  Extracting Text from the PDF

pdf_path = "../data/raw/Full-48.pdf"
pdf_name = pdf_path.split("/")[-1].split(".")[0]

output_text_file = f"../data/test/{pdf_name}.txt"

with pdfplumber.open(pdf_path) as pdf:
    extracted_text = ""
    for page in pdf.pages:
        extracted_text += page.extract_text()

with open(output_text_file, "w") as text_file:
    text_file.write(extracted_text)

print(f"Text extracted and saved to {output_text_file}")

Text extracted and saved to ../data/test/Full-48.txt


In [3]:
# Preview the Exracted Text
# reading pdf content
with open(output_text_file, "r") as file:
    document_text = file.read()

# preview the document content
print(document_text[:500])  # preview the first 500 characters

3.6: Eclipses
An eclipse is an event that happens when one body is temporarily hidden, either by passing into the shadow of another body or by
having another body pass between it and the observer.
Lunar Eclipses
A lunar eclipse occurs when Earth lies directly between the Sun and the Moon, so that Earth’s shadow falls on the Moon. A lunar
eclipse can only occur at full Moon. There are three types of lunar eclipses: Penumbral, Partial, and Total. The type of lunar eclipse
depends on how far the Mo


In [4]:
# Summarizing the Extracted Text

# Instead of:
# from transformers import pipeline
# summarizer = pipeline("summarization", model="t5-small")
# Because i do not have GPU server, unfortunately, *sad face*

import requests

API_URL = "https://router.huggingface.co/hf-inference/models/facebook/bart-large-cnn"
headers = {"Authorization": f"Bearer {hf_token}"}

def summarize(text):
    response = requests.post(API_URL, headers=headers, json={"inputs": text})
    return response.json()

result = summarize(document_text[:1024])
print("Summary:", result[0]['summary_text'])

Summary: A lunar eclipse occurs when Earth lies directly between the Sun and the Moon. There are three types of lunar eclipses: Penumbral, Partial, and Total. Total lunareclipses can vary in color and brightness due to Earth’s atmosphere. Red to Copper to Orange to Black. The colors can be quite varied and spectacular.


In [5]:
# using official Python client 
from huggingface_hub import InferenceClient

client = InferenceClient(token=hf_token)
result = client.summarization(
    document_text[:1024],
    model="facebook/bart-large-cnn"
)
print("Summary:", result.summary_text)

/root/opt/ra-i-g/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Summary: A lunar eclipse occurs when Earth lies directly between the Sun and the Moon. There are three types of lunar eclipses: Penumbral, Partial, and Total. Total lunareclipses can vary in color and brightness due to Earth’s atmosphere. Red to Copper to Orange to Black. The colors can be quite varied and spectacular.


In [6]:
# Splitting the Document into Sentences and Passages
# For more detailed analysis, like question generation, it’s important to split the document into smaller chunks. 
# This step tokenizes the document into sentences and combines them into manageable passages for subsequent steps.

import nltk
nltk.download('punkt_tab')  # not 'punkt'
from nltk.tokenize import sent_tokenize

# split text into sentences
sentences = sent_tokenize(document_text)

# combine sentences into passages
passages = []
current_passage = ""
for sentence in sentences:
    if len(current_passage.split()) + len(sentence.split()) < 200:  # adjust the word limit as needed
        current_passage += " " + sentence
    else:
        passages.append(current_passage.strip())
        current_passage = sentence
if current_passage:
    passages.append(current_passage.strip())



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [15]:
# Generating Questions from the Passages Using LLMs
# generate questions based on the document’s content. 
# This helps in understanding key information points and can be used to check the comprehension of the document. 

# using a question generation model (T5-based model valhalla/t5-base-qg-hl) from the Hugging Face transformers library to automatically 
# generate questions from text passages. 
# The function generate_questions_pipeline() takes a text passage as input and produces a list of questions. 
# We generate at least three questions for each passage, and if not, we split the passage into smaller parts and generate additional questions. 
# This approach guarantees comprehensive question generation for each passage, 
# and we print the questions along with the corresponding passage for review. 

from huggingface_hub import InferenceClient
import os

client = InferenceClient(token=os.getenv("HF_TOKEN"))

def generate_questions_pipeline(passage, min_questions=3):
    result = client.chat_completion(
        messages=[{
            "role": "user",
            "content": f"Generate exactly {min_questions} questions about the following text. Return only the questions, one per line, numbered.\n\nText: {passage}"
        }],
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        max_tokens=300
    )
    
    response = result.choices[0].message.content
    questions = [q.strip().lstrip('0123456789.-) ') for q in response.split('\n') if '?' in q]
    
    # if we didn't get enough, ask for more from a chunk
    if len(questions) < min_questions:
        passage_sentences = passage.split('. ')
        for i in range(0, len(passage_sentences), 2):
            if len(questions) >= min_questions:
                break
            chunk = '. '.join(passage_sentences[i:i+2])
            result = client.chat_completion(
                messages=[{
                    "role": "user",
                    "content": f"Generate 1 question about: {chunk}"
                }],
                model="meta-llama/Meta-Llama-3-8B-Instruct",
                max_tokens=100
            )
            q = result.choices[0].message.content.strip()
            if '?' in q:
                questions.append(q.lstrip('0123456789.-) '))
    
    return questions[:min_questions]

for idx, passage in enumerate(passages):
    questions = generate_questions_pipeline(passage)
    print(f"Passage {idx+1}:\n{passage}\n")
    print("Generated Questions:")
    for q in questions:
        print(f"- {q}")
    print(f"\n{'-'*50}\n")

Passage 1:
3.6: Eclipses
An eclipse is an event that happens when one body is temporarily hidden, either by passing into the shadow of another body or by
having another body pass between it and the observer. Lunar Eclipses
A lunar eclipse occurs when Earth lies directly between the Sun and the Moon, so that Earth’s shadow falls on the Moon. A lunar
eclipse can only occur at full Moon. There are three types of lunar eclipses: Penumbral, Partial, and Total. The type of lunar eclipse
depends on how far the Moon travels into Earth’s shadow. The entire night side of Earth can see the lunar eclipse. Total lunar
eclipses can vary in color and brightness due to Earth’s atmosphere and how deep the Moon passes into Earth’s shadow; Red to
Copper to Orange to Black. The colors can be quite varied and spectacular. A Solar Eclipse occurs when the Moon comes between the Earth and the Sun, partially or totally blocking off the Sun from view on
Earth. A solar eclipse can only occur at new Moon (whereas

In [16]:
# Answering the Generated Questions Using a QA Model. model extracts answers based on the context of the passage.

def answer_question(question, context):
    result = client.chat_completion(
        messages=[{
            "role": "user",
            "content": f"Answer the following question based only on the provided context. Give a short, direct answer.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
        }],
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        max_tokens=150
    )
    return result.choices[0].message.content.strip()

def answer_unique_questions(passages):
    answered_questions = set()
    for idx, passage in enumerate(passages):
        questions = generate_questions_pipeline(passage)
        for question in questions:
            if question not in answered_questions:
                answer = answer_question(question, passage)
                print(f"Q: {question}")
                print(f"A: {answer}\n")
                answered_questions.add(question)
        print(f"{'='*50}\n")

answer_unique_questions(passages)

Q: What is an eclipse, according to the text?
A: An eclipse is an event that happens when one body is temporarily hidden, either by passing into the shadow of another body or by having another body pass between it and the observer.

Q: What are the three types of lunar eclipses, and how do they differ?
A: The three types of lunar eclipses are Penumbral, Partial, and Total. The type of lunar eclipse depends on how far the Moon travels into Earth's shadow.

Q: When can a solar eclipse occur, according to the text?
A: According to the text, a solar eclipse can only occur at new Moon.


Q: What is the minimum requirement to see a partial solar eclipse?
A: You must be within the eclipse "zone" to see even a partial solar eclipse.

Q: What is the phase of the eclipse at which the Sun is completely covered by the Moon?
A: Totality.

Q: What phenomenon is responsible for the appearance of bands of black and white running across the ground during a total solar eclipse?
A: Earth's atmosphere.


